<a href="https://colab.research.google.com/github/tousifo/ml_notebooks/blob/main/DivideMix_(MedMNIST).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch torchvision scikit-learn torchnet tqdm medmnist

# DivideMix for Medical Image Classification (MedMNIST)

## Overview
This project implements the **DivideMix** algorithm to train robust image classification models on medical datasets containing corrupted or noisy labels. We adapted the framework to process six diverse sub-datasets from the **MedMNIST** collection, evaluating the model's resilience against varying degrees of Symmetric (random) and Asymmetric (pairwise/class-conditional) label noise.

## The DivideMix Process: How It Works


DivideMix treats the noisy label problem as a Semi-Supervised Learning (SSL) task. It uses a dual-network co-training architecture to prevent the models from memorizing the noisy labels. The training loop consists of four main phases:

1. **Warm-up Phase:** Both networks (ResNet18) are trained independently on the entire noisy dataset using standard Cross-Entropy loss for a few epochs. Deep neural networks naturally fit "clean" or easy samples faster than noisy ones, meaning clean samples will produce lower losses early in training.
2. **Loss Evaluation & GMM Fitting:** After the warm-up, the model evaluates the loss of every training sample. A two-component **Gaussian Mixture Model (GMM)** is dynamically fitted to this loss distribution to separate the clean samples from the noisy ones.
3. **Dataset Division:** The GMM calculates the probability that each sample belongs to the "low-loss" (clean) cluster.
    * Samples with a high probability are treated as **Labeled Data** (we trust their labels).
    * Samples with a low probability are stripped of their labels and treated as **Unlabeled Data**.
4. **Co-Training via MixMatch:** The divided dataset is passed to the *other* network (Network 1 divides the data for Network 2, and vice versa). The networks are then trained using **MixMatch**, a semi-supervised learning technique that uses:
    * **Label Co-guessing:** Averaging the predictions of both networks to generate artificial "pseudo-labels" for the unlabeled data.
    * **Temperature Sharpening:** Adjusting the pseudo-labels to make the model's predictions more confident.
    * **Mixup:** Blending inputs and labels together to enforce linear behavior between classes, heavily regularizing the model against overfitting to the noise.

## Experimental Setup & Modifications


We modified the baseline CIFAR-focused DivideMix framework specifically for the medical domain:

* **Datasets Evaluated:** PathMNIST, BloodMNIST, OrganAMNIST, OrganCMNIST, DermaMNIST, and OCTMNIST.
* **Image Processing:** Automated downloading, conversion of 1-channel grayscale medical scans to 3-channel RGB, and resizing to 32x32 to match the `PreResNet18` receptive field.
* **Noise Injection Profiles:**
    * **Symmetric Noise (`sym`):** Labels are randomly flipped to any other class with uniform probability (Tested at 10%, 30%, and 50% ratios).
    * **Asymmetric Noise (`asym`):** Labels are systematically flipped to the *next* adjacent class (Pairwise noise), mimicking real-world systematic annotator bias (Tested at 10%, 30%, and 50% ratios).
* **Metrics & Logging:** Replaced standard top-1 accuracy trackers with multi-class `sklearn` metrics, calculating **Accuracy**, **Macro F1-Score**, and **Multi-class AUC (One-vs-Rest)** to properly account for class imbalances in medical data.

In [2]:
%%writefile PreResNet.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable

def conv3x3(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)

class PreActBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super(PreActBlock, self).__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = conv3x3(in_planes, planes, stride)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = conv3x3(planes, planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=False)
            )
    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out)
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        out += shortcut
        return out

class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(ResNet, self).__init__()
        self.in_planes = 64
        self.conv1 = conv3x3(3,64)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.linear = nn.Linear(512*block.expansion, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x, lin=0, lout=5):
        out = x
        if lin < 1 and lout > -1:
            out = self.conv1(out)
            out = self.bn1(out)
            out = F.relu(out)
        if lin < 2 and lout > 0:
            out = self.layer1(out)
        if lin < 3 and lout > 1:
            out = self.layer2(out)
        if lin < 4 and lout > 2:
            out = self.layer3(out)
        if lin < 5 and lout > 3:
            out = self.layer4(out)
        if lout > 4:
            out = F.avg_pool2d(out, 4)
            out = out.view(out.size(0), -1)
            out = self.linear(out)
        return out

def ResNet18(num_classes=10):
    return ResNet(PreActBlock, [2,2,2,2], num_classes=num_classes)

Overwriting PreResNet.py


In [3]:
%%writefile dataloader_medmnist.py
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import random
import numpy as np
from PIL import Image
import json
import os
import torch
from torchnet.meter import AUCMeter
import medmnist
from medmnist import INFO

class medmnist_dataset(Dataset):
    def __init__(self, dataset_name, r, noise_mode, root_dir, transform, mode, noise_file='', pred=[], probability=[], log=''):
        self.r = r
        self.transform = transform
        self.mode = mode
        self.dataset_name = dataset_name

        info = INFO[dataset_name]
        self.num_classes = len(info['label'])
        DataClass = getattr(medmnist, info['python_class'])

        self.transition = {i: (i + 1) % self.num_classes for i in range(self.num_classes)}

        if self.mode == 'test':
            dataset = DataClass(split='test', download=True, root=root_dir)
            self.test_data = dataset.imgs
            self.test_label = dataset.labels.squeeze().tolist()
        else:
            dataset = DataClass(split='train', download=True, root=root_dir)
            train_data = dataset.imgs
            train_label = dataset.labels.squeeze().tolist()
            num_samples = len(train_label)

            if os.path.exists(noise_file):
                noise_label = json.load(open(noise_file,"r"))
            else:
                noise_label = []
                idx = list(range(num_samples))
                random.shuffle(idx)
                num_noise = int(self.r * num_samples)
                noise_idx = idx[:num_noise]
                for i in range(num_samples):
                    if i in noise_idx:
                        if noise_mode == 'sym':
                            noiselabel = random.randint(0, self.num_classes - 1)
                            noise_label.append(noiselabel)
                        elif noise_mode == 'asym':
                            noiselabel = self.transition[train_label[i]]
                            noise_label.append(noiselabel)
                    else:
                        noise_label.append(train_label[i])
                print("save noisy labels to %s ..." % noise_file)
                json.dump(noise_label, open(noise_file,"w"))

            if self.mode == 'all':
                self.train_data = train_data
                self.noise_label = noise_label
            else:
                if self.mode == "labeled":
                    pred_idx = pred.nonzero()[0]
                    self.probability = [probability[i] for i in pred_idx]
                    clean = (np.array(noise_label) == np.array(train_label))
                    auc_meter = AUCMeter()
                    auc_meter.reset()
                    auc_meter.add(probability, clean)
                    auc, _, _ = auc_meter.value()
                    log.write('Number of labeled samples:%d   AUC:%.3f\n' % (pred.sum(), auc))
                    log.flush()
                elif self.mode == "unlabeled":
                    pred_idx = (1 - pred).nonzero()[0]

                self.train_data = train_data[pred_idx]
                self.noise_label = [noise_label[i] for i in pred_idx]

    def __getitem__(self, index):
        if self.mode == 'labeled':
            img, target, prob = self.train_data[index], self.noise_label[index], self.probability[index]
        elif self.mode == 'unlabeled':
            img = self.train_data[index]
        elif self.mode == 'all':
            img, target = self.train_data[index], self.noise_label[index]
        elif self.mode == 'test':
            img, target = self.test_data[index], self.test_label[index]

        img = Image.fromarray(img).convert('RGB')

        if self.mode in ['labeled', 'unlabeled']:
            img1 = self.transform(img)
            img2 = self.transform(img)
            if self.mode == 'labeled': return img1, img2, target, prob
            if self.mode == 'unlabeled': return img1, img2
        elif self.mode == 'all':
            img = self.transform(img)
            return img, target, index
        elif self.mode == 'test':
            img = self.transform(img)
            return img, target

    def __len__(self):
        if self.mode != 'test':
            return len(self.train_data)
        else:
            return len(self.test_data)

class medmnist_dataloader():
    def __init__(self, dataset_name, r, noise_mode, batch_size, num_workers, root_dir, log, noise_file=''):
        self.dataset_name = dataset_name
        self.r = r
        self.noise_mode = noise_mode
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.root_dir = root_dir
        self.log = log
        self.noise_file = noise_file

        self.transform_train = transforms.Compose([
                transforms.Resize(32),
                transforms.RandomCrop(32, padding=4),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(mean=[.5, .5, .5], std=[.5, .5, .5]),
            ])
        self.transform_test = transforms.Compose([
                transforms.Resize(32),
                transforms.ToTensor(),
                transforms.Normalize(mean=[.5, .5, .5], std=[.5, .5, .5]),
            ])

    def run(self, mode, pred=[], prob=[]):
        if mode == 'warmup':
            all_dataset = medmnist_dataset(self.dataset_name, self.r, self.noise_mode, self.root_dir, self.transform_train, "all", self.noise_file)
            return DataLoader(dataset=all_dataset, batch_size=self.batch_size*2, shuffle=True, num_workers=self.num_workers)
        elif mode == 'train':
            labeled_dataset = medmnist_dataset(self.dataset_name, self.r, self.noise_mode, self.root_dir, self.transform_train, "labeled", self.noise_file, pred, prob, self.log)
            labeled_trainloader = DataLoader(dataset=labeled_dataset, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers, drop_last=True)
            unlabeled_dataset = medmnist_dataset(self.dataset_name, self.r, self.noise_mode, self.root_dir, self.transform_train, "unlabeled", self.noise_file, pred)
            unlabeled_trainloader = DataLoader(dataset=unlabeled_dataset, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers, drop_last=True)
            return labeled_trainloader, unlabeled_trainloader
        elif mode == 'test':
            test_dataset = medmnist_dataset(self.dataset_name, self.r, self.noise_mode, self.root_dir, self.transform_test, 'test')
            return DataLoader(dataset=test_dataset, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)
        elif mode == 'eval_train':
            eval_dataset = medmnist_dataset(self.dataset_name, self.r, self.noise_mode, self.root_dir, self.transform_test, 'all', self.noise_file)
            return DataLoader(dataset=eval_dataset, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

Overwriting dataloader_medmnist.py


In [4]:
%%writefile Train_medmnist.py
from __future__ import print_function
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
import random
import os
import argparse
import numpy as np
from tqdm import tqdm
from sklearn.mixture import GaussianMixture
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from PreResNet import ResNet18
import dataloader_medmnist as dataloader
from medmnist import INFO

parser = argparse.ArgumentParser(description='PyTorch MedMNIST Training - DivideMix')
parser.add_argument('--batch_size', default=64, type=int)
parser.add_argument('--lr', default=0.02, type=float)
parser.add_argument('--noise_mode', default='sym', choices=['sym', 'asym'])
parser.add_argument('--alpha', default=4, type=float)
parser.add_argument('--lambda_u', default=25, type=float)
parser.add_argument('--p_threshold', default=0.5, type=float)
parser.add_argument('--T', default=0.5, type=float)
parser.add_argument('--num_epochs', default=150, type=int)
parser.add_argument('--r', default=0.5, type=float)
parser.add_argument('--seed', default=123)
parser.add_argument('--gpuid', default=0, type=int)
parser.add_argument('--data_path', default='./medmnist_data', type=str)
parser.add_argument('--dataset', default='pathmnist', type=str,
                    choices=['pathmnist', 'bloodmnist', 'organamnist', 'organcmnist', 'dermamnist', 'octmnist'])

# parse_known_args() ignores Jupyter's background arguments, preventing errors
args, _ = parser.parse_known_args()

torch.cuda.set_device(args.gpuid)
random.seed(args.seed)
torch.manual_seed(args.seed)
torch.cuda.manual_seed_all(args.seed)

info = INFO[args.dataset]
args.num_class = len(info['label'])

def linear_rampup(current, warm_up, rampup_length=16):
    current = np.clip((current-warm_up) / rampup_length, 0.0, 1.0)
    return args.lambda_u*float(current)

class SemiLoss(object):
    def __call__(self, outputs_x, targets_x, outputs_u, targets_u, epoch, warm_up):
        probs_u = torch.softmax(outputs_u, dim=1)
        Lx = -torch.mean(torch.sum(F.log_softmax(outputs_x, dim=1) * targets_x, dim=1))
        Lu = torch.mean((probs_u - targets_u)**2)
        return Lx, Lu, linear_rampup(epoch,warm_up)

class NegEntropy(object):
    def __call__(self,outputs):
        probs = torch.softmax(outputs, dim=1)
        return torch.mean(torch.sum(probs.log()*probs, dim=1))

def train(epoch, net, net2, optimizer, labeled_trainloader, unlabeled_trainloader):
    net.train()
    net2.eval()
    unlabeled_train_iter = iter(unlabeled_trainloader)
    num_iter = (len(labeled_trainloader.dataset)//args.batch_size)+1

    pbar = tqdm(labeled_trainloader, desc=f'Train Epoch {epoch}', leave=False)
    for batch_idx, (inputs_x, inputs_x2, labels_x, w_x) in enumerate(pbar):
        try:
            inputs_u, inputs_u2 = next(unlabeled_train_iter)
        except StopIteration:
            unlabeled_train_iter = iter(unlabeled_trainloader)
            inputs_u, inputs_u2 = next(unlabeled_train_iter)
        batch_size = inputs_x.size(0)

        labels_x = torch.zeros(batch_size, args.num_class).scatter_(1, labels_x.view(-1,1), 1)
        w_x = w_x.view(-1,1).type(torch.FloatTensor)

        inputs_x, inputs_x2, labels_x, w_x = inputs_x.cuda(), inputs_x2.cuda(), labels_x.cuda(), w_x.cuda()
        inputs_u, inputs_u2 = inputs_u.cuda(), inputs_u2.cuda()

        with torch.no_grad():
            outputs_u11, outputs_u12 = net(inputs_u), net(inputs_u2)
            outputs_u21, outputs_u22 = net2(inputs_u), net2(inputs_u2)
            pu = (torch.softmax(outputs_u11, dim=1) + torch.softmax(outputs_u12, dim=1) + torch.softmax(outputs_u21, dim=1) + torch.softmax(outputs_u22, dim=1)) / 4
            ptu = pu**(1/args.T)
            targets_u = (ptu / ptu.sum(dim=1, keepdim=True)).detach()

            outputs_x, outputs_x2 = net(inputs_x), net(inputs_x2)
            px = (torch.softmax(outputs_x, dim=1) + torch.softmax(outputs_x2, dim=1)) / 2
            px = w_x*labels_x + (1-w_x)*px
            ptx = px**(1/args.T)
            targets_x = (ptx / ptx.sum(dim=1, keepdim=True)).detach()

        l = max(np.random.beta(args.alpha, args.alpha), 1 - np.random.beta(args.alpha, args.alpha))

        all_inputs = torch.cat([inputs_x, inputs_x2, inputs_u, inputs_u2], dim=0)
        all_targets = torch.cat([targets_x, targets_x, targets_u, targets_u], dim=0)
        idx = torch.randperm(all_inputs.size(0))

        mixed_input = l * all_inputs + (1 - l) * all_inputs[idx]
        mixed_target = l * all_targets + (1 - l) * all_targets[idx]

        logits = net(mixed_input)
        logits_x = logits[:batch_size*2]
        logits_u = logits[batch_size*2:]

        Lx, Lu, lamb = criterion(logits_x, mixed_target[:batch_size*2], logits_u, mixed_target[batch_size*2:], epoch+batch_idx/num_iter, warm_up)

        prior = (torch.ones(args.num_class)/args.num_class).cuda()
        pred_mean = torch.softmax(logits, dim=1).mean(0)
        penalty = torch.sum(prior*torch.log(prior/pred_mean))

        loss = Lx + lamb * Lu  + penalty
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        pbar.set_postfix({'Lx': f'{Lx.item():.2f}', 'Lu': f'{Lu.item():.2f}'})

def warmup(epoch, net, optimizer, dataloader):
    net.train()
    pbar = tqdm(dataloader, desc=f'Warmup Epoch {epoch}', leave=False)
    for batch_idx, (inputs, labels, path) in enumerate(pbar):
        inputs, labels = inputs.cuda(), labels.cuda()
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = CEloss(outputs, labels)
        if args.noise_mode=='asym':
            loss = loss + conf_penalty(outputs)
        loss.backward()
        optimizer.step()
        pbar.set_postfix({'CE-loss': f'{loss.item():.4f}'})

def eval_train(model, all_loss, eval_loader):
    model.eval()
    losses = torch.zeros(len(eval_loader.dataset))
    with torch.no_grad():
        for inputs, targets, index in tqdm(eval_loader, desc='Evaluating Loss', leave=False):
            inputs, targets = inputs.cuda(), targets.cuda()
            outputs = model(inputs)
            loss = CE(outputs, targets)
            for b in range(inputs.size(0)):
                losses[index[b]] = loss[b]
    losses = (losses-losses.min())/(losses.max()-losses.min())
    all_loss.append(losses)

    input_loss = losses.reshape(-1,1) if args.r != 0.9 else torch.stack(all_loss)[-5:].mean(0).reshape(-1,1)

    gmm = GaussianMixture(n_components=2, max_iter=10, tol=1e-2, reg_covar=5e-4)
    gmm.fit(input_loss)
    prob = gmm.predict_proba(input_loss)[:, gmm.means_.argmin()]
    return prob, all_loss

def test(epoch, net1, net2, test_loader):
    net1.eval()
    net2.eval()
    all_targets, all_outputs = [], []

    with torch.no_grad():
        for inputs, targets in tqdm(test_loader, desc='Testing', leave=False):
            inputs, targets = inputs.cuda(), targets.cuda()
            outputs = net1(inputs) + net2(inputs)
            all_targets.extend(targets.cpu().numpy())
            all_outputs.extend(torch.softmax(outputs, dim=1).cpu().numpy())

    all_targets, all_outputs = np.array(all_targets), np.array(all_outputs)
    predicted = np.argmax(all_outputs, axis=1)

    acc = accuracy_score(all_targets, predicted) * 100
    f1 = f1_score(all_targets, predicted, average='macro')
    auc = roc_auc_score(all_targets, all_outputs, multi_class='ovr') if args.num_class > 2 else roc_auc_score(all_targets, all_outputs[:, 1])

    print(f"\n| Test Epoch #{epoch}\t Accuracy: {acc:.2f}%\t F1 Score: {f1:.4f}\t AUC: {auc:.4f}\n")
    test_log.write(f'Epoch:{epoch}   Accuracy:{acc:.2f}   F1:{f1:.4f}   AUC:{auc:.4f}\n')
    test_log.flush()

if __name__ == '__main__':
    os.makedirs('./checkpoint', exist_ok=True)
    os.makedirs(args.data_path, exist_ok=True)

    stats_log = open(f'./checkpoint/{args.dataset}_{args.r}_{args.noise_mode}_stats.txt','w')
    test_log = open(f'./checkpoint/{args.dataset}_{args.r}_{args.noise_mode}_acc.txt','w')

    warm_up = 10
    noise_file_path = f'{args.data_path}/{args.dataset}_{args.r}_{args.noise_mode}.json'
    loader = dataloader.medmnist_dataloader(args.dataset, r=args.r, noise_mode=args.noise_mode, batch_size=args.batch_size, num_workers=5, root_dir=args.data_path, log=stats_log, noise_file=noise_file_path)

    print(f'| Building net for {args.dataset} (Classes: {args.num_class})')
    net1, net2 = ResNet18(num_classes=args.num_class).cuda(), ResNet18(num_classes=args.num_class).cuda()
    cudnn.benchmark = True

    criterion = SemiLoss()
    optimizer1 = optim.SGD(net1.parameters(), lr=args.lr, momentum=0.9, weight_decay=5e-4)
    optimizer2 = optim.SGD(net2.parameters(), lr=args.lr, momentum=0.9, weight_decay=5e-4)

    CE, CEloss = nn.CrossEntropyLoss(reduction='none'), nn.CrossEntropyLoss()
    if args.noise_mode == 'asym': conf_penalty = NegEntropy()
    all_loss = [[],[]]

    for epoch in range(args.num_epochs+1):
        lr = args.lr if epoch < 100 else args.lr / 10
        for param_group in optimizer1.param_groups: param_group['lr'] = lr
        for param_group in optimizer2.param_groups: param_group['lr'] = lr

        test_loader, eval_loader = loader.run('test'), loader.run('eval_train')

        if epoch < warm_up:
            warmup_trainloader = loader.run('warmup')
            warmup(epoch, net1, optimizer1, warmup_trainloader)
            warmup(epoch, net2, optimizer2, warmup_trainloader)
        else:
            prob1, all_loss[0] = eval_train(net1, all_loss[0], eval_loader)
            prob2, all_loss[1] = eval_train(net2, all_loss[1], eval_loader)

            labeled_trainloader, unlabeled_trainloader = loader.run('train', (prob2 > args.p_threshold), prob2)
            train(epoch, net1, net2, optimizer1, labeled_trainloader, unlabeled_trainloader)

            labeled_trainloader, unlabeled_trainloader = loader.run('train', (prob1 > args.p_threshold), prob1)
            train(epoch, net2, net1, optimizer2, labeled_trainloader, unlabeled_trainloader)

        test(epoch, net1, net2, test_loader)

Overwriting Train_medmnist.py


In [ ]:
datasets = ["pathmnist", "bloodmnist", "organamnist", "organcmnist", "dermamnist", "octmnist"]
modes = ["sym", "asym"]
ratios = [0.1, 0.3, 0.5]

for d in datasets:
    for m in modes:
        for r in ratios:
            print(f"\n{'='*70}")
            print(f"Training {d.upper()} | Mode: {m} | Ratio: {r} | Epochs: 9")
            print(f"{'='*70}")

            # The -u flag forces real-time unbuffered output so tqdm can render
            !python -u Train_medmnist.py --dataset {d} --noise_mode {m} --r {r} --num_epochs 15


Training PATHMNIST | Mode: sym | Ratio: 0.1 | Epochs: 9
| Building net for pathmnist (Classes: 9)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 5 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
                                              
| Test Epoch #0	 Accuracy: 77.17%	 F1 Score: 0.7308	 AUC: 0.9559

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 5 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoad